# 06 - Quantum Protocols: Teleportation and Superdense Coding

**Prerequisites:** Notebooks 01-05. Familiarity with entanglement and measurement.

Quantum teleportation and superdense coding are two of the most celebrated
protocols in quantum information. They reveal the power of **entanglement as a
resource** -- something that has no classical analogue.

- **Quantum teleportation** transmits an unknown quantum state from Alice to Bob
  using one shared Bell pair and two classical bits. No physical qubit travels
  between them; the quantum information is reconstructed at the destination.

- **Superdense coding** does the reverse: Alice sends two classical bits of
  information to Bob by transmitting only one qubit, again leveraging a
  pre-shared Bell pair.

Both protocols require **dynamic circuits** -- mid-circuit measurement followed
by classically conditioned gates (feed-forward).

By the end of this notebook you will be able to:

1. **Implement** the quantum teleportation protocol with mid-circuit measurement.
2. **Implement** superdense coding to send 2 classical bits via 1 qubit.
3. **Explain** the role of classical communication in both protocols.
4. **Verify** that teleportation preserves the input state's measurement statistics.

In this notebook we will:

1. Build the full quantum teleportation circuit and verify it works.
2. Understand the role of classical communication.
3. Implement superdense coding for all four 2-bit messages.
4. Use Goqu's `If`, `Measure`, and `RunDynamic` for feed-forward control.

In [1]:
import (
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/gate"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## The Quantum Teleportation Protocol

Suppose Alice has a qubit in an unknown state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ and
wants to transmit it to Bob. She cannot simply measure it (that would destroy the
superposition), and the no-cloning theorem says she cannot copy it. But with a
shared Bell pair and two classical bits, she can teleport the state perfectly.

### The protocol step by step

**Setup:** Alice and Bob share a Bell pair $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$.
Alice holds qubit 1 (her half) and Bob holds qubit 2.

**Step 1 -- Alice's operations:**
- Alice applies CNOT with her data qubit (q0) as control and her Bell qubit (q1) as target.
- Alice applies H to her data qubit (q0).

**Step 2 -- Alice measures:**
- Alice measures both her qubits (q0 and q1), obtaining two classical bits.

**Step 3 -- Bob's corrections (classically conditioned):**
- If Alice's measurement of q1 is 1, Bob applies X to his qubit (q2).
- If Alice's measurement of q0 is 1, Bob applies Z to his qubit (q2).

After these corrections, Bob's qubit is in exactly the state $|\psi\rangle$.

The circuit uses 3 qubits and 2 classical bits:
- q0: Alice's data qubit (the state to teleport)
- q1: Alice's half of the Bell pair
- q2: Bob's half of the Bell pair
- c0: result of measuring q0
- c1: result of measuring q1

In [2]:
%%
// Build the full teleportation circuit.
// We prepare an interesting state on q0 using RY(pi/3),
// which gives |psi> = cos(pi/6)|0> + sin(pi/6)|1> = (sqrt(3)/2)|0> + (1/2)|1>.

cTeleport, err := builder.New("teleport", 3).WithClbits(2).
	// Prepare the state to teleport on q0
	RY(math.Pi/3, 0).
	// Create Bell pair between q1 (Alice) and q2 (Bob)
	H(1).CNOT(1, 2).
	// Alice's operations: entangle q0 with q1
	CNOT(0, 1).H(0).
	// Alice measures both her qubits
	Measure(0, 0).  // q0 -> c0
	Measure(1, 1).  // q1 -> c1
	// Bob's corrections conditioned on Alice's results
	If(1, 1, gate.X, 2).  // if c1 == 1, apply X to q2
	If(0, 1, gate.Z, 2).  // if c0 == 1, apply Z to q2
	Build()
if err != nil {
	panic(err)
}

fmt.Println("Quantum Teleportation Circuit:")
gonbui.DisplayHTML(draw.SVG(cTeleport))
fmt.Printf("Is dynamic circuit: %v\n", cTeleport.IsDynamic())

// Show the input state we are teleporting
angle := math.Pi / 3
fmt.Printf("\nInput state: RY(pi/3)|0> = %.4f|0> + %.4f|1>\n",
	math.Cos(angle/2), math.Sin(angle/2))
fmt.Printf("Expected probabilities: P(0) = %.4f, P(1) = %.4f\n",
	math.Cos(angle/2)*math.Cos(angle/2),
	math.Sin(angle/2)*math.Sin(angle/2))

Quantum Teleportation Circuit:
Is dynamic circuit: true

Input state: RY(pi/3)|0> = 0.8660|0> + 0.5000|1>
Expected probabilities: P(0) = 0.7500, P(1) = 0.2500


q0 
 
 q1 
 
 q2 
 
 RY(pi/3) 
 H 
 
 
 
 
 
 
 H 
 M 
 M 
 X 
 Z

## Verifying Teleportation Works

To verify that teleportation succeeded, we add a final measurement on Bob's
qubit (q2) and check that the measurement statistics match the original input
state.

The input state $\text{RY}(\pi/3)|0\rangle$ has:
- $P(0) = \cos^2(\pi/6) = 3/4 = 0.75$
- $P(1) = \sin^2(\pi/6) = 1/4 = 0.25$

If teleportation works correctly, Bob's qubit should show these same
probabilities regardless of Alice's measurement outcomes.

In [3]:
%%
// Build teleportation circuit with a final measurement on Bob's qubit.
// We use 3 classical bits: c0 and c1 for Alice, c2 for Bob's result.
cVerify, err := builder.New("teleport-verify", 3).WithClbits(3).
	// Prepare state to teleport
	RY(math.Pi/3, 0).
	// Bell pair
	H(1).CNOT(1, 2).
	// Alice's operations
	CNOT(0, 1).H(0).
	// Alice measures
	Measure(0, 0).Measure(1, 1).
	// Bob corrects
	If(1, 1, gate.X, 2).
	If(0, 1, gate.Z, 2).
	// Bob measures his qubit
	Measure(2, 2).
	Build()
if err != nil {
	panic(err)
}

sim := statevector.New(3)
counts, err := sim.RunDynamic(cVerify, 4000)
if err != nil {
	panic(err)
}

// Extract Bob's qubit statistics (c2 is the most-significant bit in the bitstring)
bob0, bob1 := 0, 0
for bs, n := range counts {
	// Bitstring format is [c2 c1 c0] -- c2 is the leftmost character
	if bs[0] == '0' {
		bob0 += n
	} else {
		bob1 += n
	}
}

total := bob0 + bob1
fmt.Println("Teleportation verification (4000 shots):")
fmt.Println("\nAll outcomes [c2 c1 c0]:")
for bs, n := range counts {
	fmt.Printf("  %s: %d\n", bs, n)
}
fmt.Println("\nBob's qubit (q2) statistics:")
fmt.Printf("  P(0) = %d/%d = %.4f  (expected: 0.7500)\n", bob0, total, float64(bob0)/float64(total))
fmt.Printf("  P(1) = %d/%d = %.4f  (expected: 0.2500)\n", bob1, total, float64(bob1)/float64(total))
fmt.Println("\nTeleportation successful: Bob's statistics match the input state!")

// Visualize Bob's measurement results
bobCounts := map[string]int{"0": bob0, "1": bob1}
gonbui.DisplayHTML(viz.Histogram(bobCounts, viz.WithTitle("Bob's Qubit After Teleportation")))

Teleportation verification (4000 shots):

All outcomes [c2 c1 c0]:
  010: 768
  101: 259
  111: 234
  001: 780
  110: 235
  011: 689
  000: 792
  100: 243

Bob's qubit (q2) statistics:
  P(0) = 3029/4000 = 0.7572  (expected: 0.7500)
  P(1) = 971/4000 = 0.2427  (expected: 0.2500)

Teleportation successful: Bob's statistics match the input state!


Bob's Qubit After Teleportation 
 
 
 
 0 
 
 500 
 
 1000 
 
 1500 
 
 2000 
 
 2500 
 
 3000 
 
 3500 
 
 0 
 
 1

## Superdense Coding

Superdense coding is the "dual" protocol to teleportation. While teleportation
sends one qubit of quantum information using two classical bits and one shared
ebit, superdense coding sends **two classical bits** using one qubit and one
shared ebit.

### The protocol

**Setup:** Alice and Bob share a Bell pair $|\Phi^+\rangle$. Alice holds qubit 0, Bob holds qubit 1.

**Encoding:** To send a 2-bit message, Alice applies an operation to *only her qubit*:

| Message | Alice applies | Resulting Bell state |
|:---:|:---:|:---:|
| 00 | I (nothing) | $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ |
| 01 | Z | $|\Phi^-\rangle = \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle)$ |
| 10 | X | $|\Psi^+\rangle = \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle)$ |
| 11 | X then Z | $|\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$ |

**Decoding:** Alice sends her qubit to Bob. Bob now has both halves of the
(modified) Bell pair. He applies CNOT(0,1) then H(0) and measures both qubits
to recover the original 2-bit message.

This works because the four Bell states are orthogonal -- Bob can perfectly
distinguish them with the right measurement.

In [4]:
%%
// Demonstrate superdense coding for all four 2-bit messages.

messages := []struct {
	bits string
	desc string
}{
	{"00", "I (identity)"},
	{"01", "Z"},
	{"10", "X"},
	{"11", "X then Z"},
}

for _, msg := range messages {
	// Build the superdense coding circuit
	b := builder.New("superdense-"+msg.bits, 2)

	// Step 1: Create shared Bell pair
	b.H(0).CNOT(0, 1)

	// Step 2: Alice encodes her 2-bit message on her qubit (q0)
	switch msg.bits {
	case "00":
		// Apply nothing (identity)
	case "01":
		b.Z(0)
	case "10":
		b.X(0)
	case "11":
		b.X(0).Z(0)
	}

	// Step 3: Bob decodes -- CNOT then H, then measure
	b.CNOT(0, 1).H(0).MeasureAll()

	c, err := b.Build()
	if err != nil {
		panic(err)
	}

	fmt.Printf("=== Message: %s (Alice applies %s) ===\n", msg.bits, msg.desc)
	gonbui.DisplayHTML(draw.SVG(c))

	sim := statevector.New(2)
	counts, err := sim.Run(c, 1000)
	if err != nil {
		panic(err)
	}

	fmt.Printf("Bob decodes: %v\n\n", counts)
}

=== Message: 00 (Alice applies I (identity)) ===
Bob decodes: map[00:1000]

=== Message: 01 (Alice applies Z) ===


q0 
 
 q1 
 
 H 
 
 
 
 
 
 
 H 
 M 
 M

q0 
 
 q1 
 
 H 
 
 
 
 Z 
 
 
 
 H 
 M 
 M

Bob decodes: map[01:1000]

=== Message: 10 (Alice applies X) ===
Bob decodes: map[10:1000]

=== Message: 11 (Alice applies X then Z) ===


q0 
 
 q1 
 
 H 
 
 
 
 X 
 
 
 
 H 
 M 
 M

q0 
 
 q1 
 
 H 
 
 
 
 X 
 Z 
 
 
 
 H 
 M 
 M

Bob decodes: map[11:1000]



## Predict, Then Verify

**Question:** In the superdense coding protocol, what if Alice wants to send the
message **11**? She applies X then Z to her qubit. What will Bob measure?

**Pause and predict** before reading further.

*Hint: After X then Z on Alice's qubit, the Bell pair becomes $|\Psi^-\rangle$. Bob applies CNOT(0,1) then H(0) to decode.*

In [5]:
%%
// Verify the prediction: superdense coding with message 11
cSD11, err := builder.New("superdense-11", 2).
	// Bell pair
	H(0).CNOT(0, 1).
	// Alice encodes 11: X then Z
	X(0).Z(0).
	// Bob decodes
	CNOT(0, 1).H(0).
	MeasureAll().
	Build()
if err != nil {
	panic(err)
}

fmt.Println("Superdense coding: Alice sends message 11")
gonbui.DisplayHTML(draw.SVG(cSD11))

sim := statevector.New(2)
counts, err := sim.Run(cSD11, 1000)
if err != nil {
	panic(err)
}

fmt.Printf("Bob's measurement: %v\n", counts)
if counts["11"] == 1000 {
	fmt.Println("\nPrediction confirmed: Bob always measures 11!")
}

gonbui.DisplayHTML(viz.Histogram(counts, viz.WithTitle("Superdense Coding: Message 11")))

Superdense coding: Alice sends message 11


q0 
 
 q1 
 
 H 
 
 
 
 X 
 Z 
 
 
 
 H 
 M 
 M

Bob's measurement: map[11:1000]

Prediction confirmed: Bob always measures 11!


Superdense Coding: Message 11 
 
 
 
 0 
 
 200 
 
 400 
 
 600 
 
 800 
 
 1000 
 
 11

## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. In teleportation, does any quantum information travel through the classical channel? Why or why not?
2. How many classical bits does superdense coding send per shared Bell pair?
3. Why do both teleportation and superdense coding require classical communication in addition to entanglement?

## Exercises

### Exercise 1: Teleport an Arbitrary State

Modify the teleportation circuit to teleport a different state. Use
`RY(theta, 0)` with a different angle (e.g., $\pi/4$ or $2\pi/3$) and verify
that Bob's measurement statistics match the expected probabilities of the
input state.

*Hint:* For $\text{RY}(\theta)|0\rangle$, the probabilities are
$P(0) = \cos^2(\theta/2)$ and $P(1) = \sin^2(\theta/2)$.

In [6]:
%%
// Exercise 1: Teleport RY(2*pi/3)|0>
// Expected: P(0) = cos^2(pi/3) = 0.25, P(1) = sin^2(pi/3) = 0.75
//
// TODO: Choose a different angle for RY and build the teleportation circuit.
// Follow the same pattern as the example above:
//   1. Prepare the state with RY(theta, 0)
//   2. Create a Bell pair on qubits 1 and 2
//   3. Alice's operations: CNOT(0,1), H(0)
//   4. Alice measures q0 -> c0, q1 -> c1
//   5. Bob corrects: If c1==1 apply X to q2, If c0==1 apply Z to q2
//   6. Bob measures q2 -> c2

theta := 2.0 * math.Pi / 3.0

// cEx1, err := builder.New("teleport-ex1", 3).WithClbits(3).
// 	RY(theta, 0).                    // Prepare state to teleport
// 	/* TODO: Bell pair */
// 	/* TODO: Alice's operations */
// 	/* TODO: Alice measures */
// 	/* TODO: Bob corrects */
// 	Measure(2, 2).                   // Bob measures
// 	Build()
// if err != nil {
// 	panic(err)
// }
//
// fmt.Println(draw.String(cEx1))
//
// simEx1 := statevector.New(3)
// countsEx1, err := simEx1.RunDynamic(cEx1, 4000)
// if err != nil {
// 	panic(err)
// }
//
// // TODO: Extract Bob's statistics from countsEx1
// // Hint: The bitstring format is [c2 c1 c0], so c2 is bs[0]
// // b0, b1 := 0, 0
// // for bs, n := range countsEx1 { ... }
//
// fmt.Printf("Expected: P(0) = %.4f, P(1) = %.4f\n",
// 	math.Cos(theta/2)*math.Cos(theta/2),
// 	math.Sin(theta/2)*math.Sin(theta/2))

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")
_ = theta

TODO: Uncomment the code above and fill in the missing parts!


### Exercise 2: Superdense Coding -- Verify All Four Messages

Build a loop that encodes each of the four 2-bit messages (00, 01, 10, 11)
and verifies that Bob always decodes the correct message with certainty.
Display all results in a single summary.

In [7]:
%%
// Exercise 2: Verify superdense coding for all 4 messages
//
// TODO: Build a loop over all four 2-bit messages (00, 01, 10, 11).
// For each message:
//   1. Create a Bell pair: H(0), CNOT(0,1)
//   2. Alice encodes by applying X and/or Z to q0 based on the message
//   3. Bob decodes: CNOT(0,1), H(0), MeasureAll
//   4. Run the circuit and check that Bob always gets the correct message
//
// Hint: 00 -> nothing, 01 -> X, 10 -> Z, 11 -> X then Z

// type sdMessage struct {
// 	bits   string
// 	applyX bool
// 	applyZ bool
// }
//
// sdMessages := []sdMessage{
// 	{"00", false, false},
// 	{"01", true, false},
// 	{"10", false, true},
// 	{"11", true, true},
// }
//
// allCorrect := true
// fmt.Println("Superdense Coding Verification")
// fmt.Println("==============================")
//
// for _, msg := range sdMessages {
// 	b := builder.New("sd-"+msg.bits, 2)
// 	// TODO: Build Bell pair
// 	// TODO: Alice encodes based on msg.applyX and msg.applyZ
// 	// TODO: Bob decodes: CNOT, H, MeasureAll
//
// 	c, err := b.Build()
// 	if err != nil {
// 		panic(err)
// 	}
//
// 	sim := statevector.New(2)
// 	counts, err := sim.Run(c, 1000)
// 	if err != nil {
// 		panic(err)
// 	}
//
// 	result := "PASS"
// 	if counts[msg.bits] != 1000 {
// 		result = "FAIL"
// 		allCorrect = false
// 	}
// 	fmt.Printf("  Message %s -> Bob decoded: %v  [%s]\n", msg.bits, counts, result)
// }
//
// if allCorrect {
// 	fmt.Println("\nAll four messages decoded correctly!")
// }

fmt.Println("TODO: Uncomment the code above and fill in the missing parts!")

TODO: Uncomment the code above and fill in the missing parts!


## Key Takeaways

1. **Quantum teleportation** transmits an unknown quantum state using one
   shared Bell pair and two classical bits. The quantum state is destroyed
   at the source and recreated at the destination -- no cloning occurs.

2. **Superdense coding** sends two classical bits by transmitting one qubit,
   using a pre-shared Bell pair. It is the "dual" of teleportation.

3. Both protocols require **entanglement as a resource**. Without the shared
   Bell pair, neither protocol is possible.

4. Both protocols require **classical communication**. Teleportation needs
   Alice to send her 2-bit measurement result to Bob; superdense coding
   needs Alice to physically send her qubit. Neither violates the
   no-communication theorem or enables faster-than-light signaling.

5. **Dynamic circuits** with mid-circuit measurement and classically
   conditioned gates (feed-forward) are essential for implementing
   teleportation. Goqu supports this via `Measure`, `If`, and `RunDynamic`.

6. The four **Bell states** form a complete basis that enables both protocols.
   In teleportation, Alice's measurement projects onto the Bell basis;
   in superdense coding, Alice encodes by rotating between Bell states.